In [1]:
homedir = '/u/az6922/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [2]:
stime = 144 # ms
nlinks = 2048
nhosts = 3072
bw = 1342176000 # B per second
load_list = range(1,10) # [10,30,50,70]
seed_list = [1,2,3,4,5]
topologytype = 1
nswitches = 80
os = 1
k = 64
nintervals = 1
npfile = 'evalnetpathfiles/netpath_leafspine_80_64_ecmp.np'
pwfile = 'experiments/nsdi26fall/eval_main/prv1/pwfiles/pathweight_leafspine.pw'

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/)
cp eval_main/unv1/pwfiles/pathweight_leafspine.pw eval_main/prv1/pwfiles/pathweight_leafspine.pw

In [3]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/prv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 222240244500, maxinterval 7, fullload 395823808512.0, ratio 1.7810626936742773


In [4]:
# generate connection_matrices file (2)
random.seed(0)
for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/leafspine_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 1%, totalbytes 3958238085.12, unv1bytes 222240244500, mult 0.017810626936742773, actualbytes 3958243500
load 2%, totalbytes 7916476170.24, unv1bytes 222240244500, mult 0.035621253873485546, actualbytes 7916485500
load 3%, totalbytes 11874714255.36, unv1bytes 222240244500, mult 0.05343188081022832, actualbytes 11874876000
load 4%, totalbytes 15832952340.48, unv1bytes 222240244500, mult 0.07124250774697109, actualbytes 15833023500
load 5%, totalbytes 19791190425.6, unv1bytes 222240244500, mult 0.08905313468371386, actualbytes 19791190500
load 6%, totalbytes 23749428510.72, unv1bytes 222240244500, mult 0.10686376162045665, actualbytes 23749429500
load 7%, totalbytes 27707666595.84, unv1bytes 222240244500, mult 0.12467438855719941, actualbytes 27707725500
load 8%, totalbytes 31665904680.96, unv1bytes 222240244500, mult 0.14248501549394219, actualbytes 31665991500
load 9%, totalbytes 35624142766.08, unv1bytes 222240244500, mult 0.16029564243068498, actualbytes 35624212500


In [5]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/prv1/run_ls.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/prv1/cmfiles/leafspine_load{load}.cm'
            outfile = f'experiments/nsdi26fall/eval_main/prv1/outfiles/leafspine_load{load}_seed{seed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfile} -numintervals {nintervals} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/prv1/run_ls.conf --worker 32